In [ ]:
import pandas as pd
import numpy as np
import torch
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)
from datasets import Dataset
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score
from sklearn.model_selection import train_test_split
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
import time
import json
import random
from tqdm.auto import tqdm


In [ ]:
# Доли пропущенных признаков для экспериментов
MISSING_RATES = [0.0, 0.2, 0.5, 0.9]

# Настройка промпта для Credit_G
prompt_config = {
        "task": "Classify credit risk as good or bad",
        "pos_label": "good",
        "neg_label": "bad",
        "entity": "Client",
        "question": "Is this client a good credit risk?"
}

openml_id = 31

base_output_dir = "/content/drive/MyDrive/finetuned_qwen_missing_Credit_G"

# 1. Загрузка данных

In [ ]:
import pandas as pd
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

def load_dataset(openml_id=1464):
    dataset = fetch_openml(data_id=openml_id, as_frame=True, parser='auto')
    X = dataset.data
    y = dataset.target

    df = X.copy()
    feature_names = X.columns.to_list()

    target_name = y.name
    df[target_name] = y

    # Преобразование целевой переменной в бинарный формат
    if y.dtype == 'object' or y.dtype.name == 'category':
        le = LabelEncoder()
        df[target_name] = le.fit_transform(df[target_name])

    return df, feature_names, target_name


def split_dataset(df, target_name, test_size=0.2, val_size=0.25, seed=42):
    # Разделение на train/val/test (60/20/20)
    train_val_df, test_df = train_test_split(
        df,
        test_size=test_size,
        random_state=seed,
        stratify=df[target_name]
    )

    train_df, val_df = train_test_split(
        train_val_df,
        test_size=val_size,
        random_state=seed,
        stratify=train_val_df[target_name]
    )

    return train_df, val_df, test_df

df, feature_names, target_name = load_dataset(openml_id)
train_df, val_df, test_df = split_dataset(df, target_name)

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 21 columns):
 #   Column                  Non-Null Count  Dtype   
---  ------                  --------------  -----   
 0   checking_status         1000 non-null   category
 1   duration                1000 non-null   int64   
 2   credit_history          1000 non-null   category
 3   purpose                 1000 non-null   category
 4   credit_amount           1000 non-null   int64   
 5   savings_status          1000 non-null   category
 6   employment              1000 non-null   category
 7   installment_commitment  1000 non-null   int64   
 8   personal_status         1000 non-null   category
 9   other_parties           1000 non-null   category
 10  residence_since         1000 non-null   int64   
 11  property_magnitude      1000 non-null   category
 12  age                     1000 non-null   int64   
 13  other_payment_plans     1000 non-null   category
 14  housing                 1

In [ ]:
for name, df in [("train", train_df), ("val", val_df), ("test", test_df)]:
    counts = df[target_name].value_counts()
    pcts   = df[target_name].value_counts(normalize=True) * 100
    print(f"\n{name} (всего: {len(df)}):")
    print(f"  {prompt_config['neg_label']}: {counts[0]:5d} — {pcts[0]:.1f}%")
    print(f"  {prompt_config['pos_label']}: {counts[1]:5d} — {pcts[1]:.1f}%")


train (всего: 600):
  bad:   180 — 30.0%
  good:   420 — 70.0%

val (всего: 200):
  bad:    60 — 30.0%
  good:   140 — 70.0%

test (всего: 200):
  bad:    60 — 30.0%
  good:   140 — 70.0%


# 2. Вспомогательные функции

In [ ]:
def row_to_text_template_with_missing(row, feature_names, target_name, missing_rate=0.0, seed=None):

    rng = np.random.default_rng(seed)

    # Определяем, какие признаки пропустить
    n_drop = int(len(feature_names) * missing_rate)
    dropped = set(rng.choice(feature_names, size=n_drop, replace=False)) if n_drop > 0 else set()

    template_parts = []

    for feature in feature_names:
        if feature in dropped:
            continue  # пропускаем признак

        value = row[feature]

        if isinstance(value, (int, np.integer)):
            phrase = f"The value of {feature} is {value}."
        elif isinstance(value, (float, np.floating)):
            phrase = f"The value of {feature} is {value:.2f}."
        else:
            phrase = f"The category of {feature} is {value}."

        template_parts.append(phrase)

    text = " ".join(template_parts)

    return text

# Тест
test_row = train_df.iloc[0]
display(train_df.head(1))
for rate in MISSING_RATES:
    text_output = row_to_text_template_with_missing(
        row=test_row,
        feature_names=feature_names,
        target_name=target_name,
        missing_rate=rate
    )

    print(f"Missing Rate: {rate * 100:.0f}%")
    print(text_output)

,checking_status,duration,credit_history,purpose,credit_amount,savings_status,employment,installment_commitment,personal_status,other_parties,...,property_magnitude,age,other_payment_plans,housing,existing_credits,job,num_dependents,own_telephone,foreign_worker,class
251,>=200,12,existing paid,furniture/equipment,2251,<100,1<=X<4,1,female div/dep/mar,none,...,car,46,none,own,1,unskilled resident,1,none,yes,1


Missing Rate: 0%
The category of checking_status is >=200. The value of duration is 12. The category of credit_history is existing paid. The category of purpose is furniture/equipment. The value of credit_amount is 2251. The category of savings_status is <100. The category of employment is 1<=X<4. The value of installment_commitment is 1. The category of personal_status is female div/dep/mar. The category of other_parties is none. The value of residence_since is 2. The category of property_magnitude is car. The value of age is 46. The category of other_payment_plans is none. The category of housing is own. The value of existing_credits is 1. The category of job is unskilled resident. The value of num_dependents is 1. The category of own_telephone is none. The category of foreign_worker is yes.
Missing Rate: 20%
The category of checking_status is >=200. The category of purpose is furniture/equipment. The value of credit_amount is 2251. The category of savings_status is <100. The value o

In [ ]:
def parse_prediction(response, prompt_config):
    response = response.lower().strip().rstrip('.,!? ')
    pos, neg = prompt_config['pos_label'].lower(), prompt_config['neg_label'].lower()

    # Точное совпадение
    if response == pos: return 1
    if response == neg: return 0

    # Начинается с метки
    if response.startswith(pos): return 1
    if response.startswith(neg): return 0

    # Содержит метку как отдельное слово
    words = response.split()
    if pos in words: return 1
    if neg in words: return 0

    # Не удалось распознать
    print(f"Warning: Could not parse '{response}' (expected '{prompt_config['pos_label']}' or '{prompt_config['neg_label']}')")
    return 0



response = prompt_config['pos_label']
pred = parse_prediction(response, prompt_config)
print(f"Response: '{response}'\nPrediction: {pred}\n")

response = "hi"
pred = parse_prediction(response, prompt_config)
print(f"Response: '{response}'\nPrediction: {pred}")

Response: 'good'
Prediction: 1

Response: 'hi'
Prediction: 0


In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
# Вычисление метрик качества
def compute_metrics(y_true, y_pred, y_prob=None):
    roc = roc_auc_score(y_true, y_prob if y_prob is not None else y_pred)
    return {
        "ROC-AUC": roc_auc_score(y_true, y_prob if y_prob is not None else y_pred),
        "F1":       f1_score(y_true, y_pred, zero_division=0),
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall":   recall_score(y_true, y_pred, zero_division=0),
    }

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.utils import resample

def bootstrap_metrics(y_true, y_pred, y_prob=None, n_iter=1000):
    scores = []

    for i in range(n_iter):
        # Bootstrap выборка
        if y_prob is not None:
            y_true_boot, y_pred_boot, y_prob_boot = resample(
                y_true, y_pred, y_prob, random_state=i+1
            )
        else:
            y_true_boot, y_pred_boot = resample(
                y_true, y_pred, random_state=i+1
            )
            y_prob_boot = None

        try:
            # Вычисление метрик
            if y_prob_boot is not None:
                auc = roc_auc_score(y_true_boot, y_prob_boot)
            else:
                auc = 0.0

            f1 = f1_score(y_true_boot, y_pred_boot, zero_division=0)
            pr = precision_score(y_true_boot, y_pred_boot, zero_division=0)
            rc = recall_score(y_true_boot, y_pred_boot, zero_division=0)

            acc = accuracy_score(y_true_boot, y_pred_boot)
            scores.append((auc, f1, acc, pr, rc))

        except ValueError:
            # Пропускание сэмплов, где не представлены все классы
            continue

    scores = np.asarray(scores)
    means, stds = scores.mean(0), scores.std(0, ddof=1)
    names = ["ROC-AUC", "F1", "Accuracy", "Precision", "Recall"]

    return {n: f"{m:.4f}±{s:.4f}" for n, m, s in zip(names, means, stds)}

# Тестирование функции
sample_row = test_df.iloc[0]
sample_text = row_to_text_template_with_missing(sample_row, feature_names, target_name)
print(f"\nПример текста:\n{sample_text[:300]}")


Пример текста:
The category of checking_status is 0<=X<200. The value of duration is 18. The category of credit_history is delayed previously. The category of purpose is business. The value of credit_amount is 2427. The category of savings_status is no known savings. The category of employment is >=7. The value of


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

#Загрузка модели Qwen 3.0 4B Instruct
model_name = "Qwen/Qwen3-4B-Instruct-2507"

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# Загрузка токенизаторов
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Загрузка модель с 16-битной квантизацией и автоматическим распределением по устройствам
base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype=torch.bfloat16,
    device_map=device, # Explicitly force all model layers to the detected device (GPU if available)
    attn_implementation="sdpa"
)

if torch.cuda.is_available():
    print(f"Использовано памяти: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

Using device: cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/238 [00:00<?, ?B/s]

Использовано памяти: 8.04 GB


In [ ]:
def create_prompt_missing(row, feature_names, target_name, prompt_config, tokenizer, missing_rate=0.0, seed=None):

    system_prompt = (
        f"You are a classifier. {prompt_config['task']}: "
        f"'{prompt_config['pos_label']}' or '{prompt_config['neg_label']}'. "
        f"Answer with only one word: '{prompt_config['pos_label']}' or '{prompt_config['neg_label']}'."
    )

    input_text = row_to_text_template_with_missing(row, feature_names, target_name, missing_rate, seed)

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user",
         "content": f"{prompt_config['entity']} information: {input_text}\n{prompt_config['question']}"},
    ]

    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True # добавление role assistant
    )

In [ ]:
import torch.nn.functional as F

def predict_single_with_prob(prompt, prompt_config, model, tokenizer, device, max_new_tokens=5):

    model_inputs = tokenizer([prompt], return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = model.generate(
            model_inputs.input_ids,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
            output_scores=True,
            return_dict_in_generate=True
        )

    # Декодирование текста
    generated_ids = outputs.sequences[0][model_inputs.input_ids.shape[1]:]
    response = tokenizer.decode(generated_ids, skip_special_tokens=True).strip().lower()

    # Извлечение вероятностей для меток из prompt_config
    first_token_logits = outputs.scores[0][0]

    pos_id = tokenizer.encode(prompt_config['pos_label'], add_special_tokens=False)[0]
    neg_id = tokenizer.encode(prompt_config['neg_label'], add_special_tokens=False)[0]

    pos_logit = first_token_logits[pos_id]
    neg_logit = first_token_logits[neg_id]

    probs = F.softmax(torch.stack([pos_logit, neg_logit]), dim=0)
    prob_pos = probs[0].item()

    return response, prob_pos

# Тест
test_prompt = create_prompt_missing(test_df.iloc[0], feature_names, target_name, prompt_config, tokenizer)
response, prob = predict_single_with_prob(test_prompt, prompt_config, base_model, tokenizer, device)
print(f"Response: '{response}', Probability: {prob:.4f}")

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Response: 'bad', Probability: 0.0000


# 3. Fine-tuning с LoRA

## Подготовка данных для Fine-tuning

In [ ]:
def balance_dataset(train_df, target_name, prompt_config, seed=42):
    df_0 = train_df[train_df[target_name] == 0]
    df_1 = train_df[train_df[target_name] == 1]

    df_majority = df_0 if len(df_0) > len(df_1) else df_1
    df_minority = df_1 if len(df_0) > len(df_1) else df_0

    print(f"До балансировки:")
    print(f"{prompt_config['neg_label']}:  {len(df_majority)}")
    print(f"{prompt_config['pos_label']}: {len(df_minority)}")

    df_minority_up = resample(df_minority, replace=True,
                              n_samples=len(df_majority), random_state=seed)

    train_df_balanced = pd.concat([df_majority, df_minority_up])
    train_df_balanced = train_df_balanced.sample(frac=1, random_state=seed).reset_index(drop=True)

    print(f"\nПосле балансировки:")
    print(train_df_balanced[target_name].value_counts())

    return train_df_balanced

train_df_balanced = balance_dataset(train_df, target_name, prompt_config)

До балансировки:
bad:  420
good: 180

После балансировки:
class
0    420
1    420
Name: count, dtype: int64


In [ ]:
# Создание текстов для обучения

def create_training_dataset(df, feature_names, target_name, prompt_config, tokenizer, missing_rate=0.0):
    system_prompt = (
        f"You are a classifier. {prompt_config['task']}: "
        f"'{prompt_config['pos_label']}' or '{prompt_config['neg_label']}'. "
        f"Answer with only one word: '{prompt_config['pos_label']}' or '{prompt_config['neg_label']}'."
    )
    texts = []

    for idx, (_, row) in enumerate(tqdm(df.iterrows(), total=len(df), desc=f"Dataset (missing={missing_rate:.0%})")):
        input_text = row_to_text_template_with_missing(row, feature_names, target_name, missing_rate, idx)
        target = prompt_config['pos_label'] if row[target_name] == 1 else prompt_config['neg_label']

        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": f"{prompt_config['entity']} information: {input_text}\n{prompt_config['question']}"},
            {"role": "assistant", "content": target}
        ]

        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False
        )
        texts.append(text)

    return texts


In [ ]:
def finetune_model(missing_rate, train_df_balanced, feature_names, target_name,
                   prompt_config, tokenizer, base_model, device,
                   num_epochs=3, batch_size=16):
    """
    Полный цикл fine-tuning для заданного missing_rate.
    Возвращает путь к лучшему checkpoint.
    """
    output_dir = f"{base_output_dir}_missing{int(missing_rate * 100):03d}"
    print(f"  Fine-tuning: missing_rate = {missing_rate:.0%}")
    print(f"  Output dir : {output_dir}")

    # Датасет
    train_texts = create_training_dataset(
        train_df_balanced, feature_names, target_name,
        prompt_config, tokenizer, missing_rate=missing_rate)
    train_dataset = Dataset.from_dict({"text": train_texts})

    def tokenize_fn(examples):
        return tokenizer(examples["text"], truncation=True, max_length=512, padding=False)

    tokenized_dataset = train_dataset.map(tokenize_fn, batched=True,
                                          remove_columns=train_dataset.column_names)

    # LoRA
    lora_config = LoraConfig(
        r=16, lora_alpha=32,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                        "gate_proj", "up_proj", "down_proj"],
        lora_dropout=0.05, bias="none", task_type="CAUSAL_LM"
    )
    model_lora = get_peft_model(base_model, lora_config)
    model_lora.print_trainable_parameters()

    training_args = TrainingArguments(
        output_dir=output_dir,
        num_train_epochs=num_epochs,
        per_device_train_batch_size=batch_size,
        gradient_accumulation_steps=2,
        learning_rate=2e-4,
        bf16=True,
        logging_steps=10,
        save_strategy="epoch",
        optim="adamw_torch_fused",
        warmup_steps=50,
        max_grad_norm=1.0,
        weight_decay=0.01,
        report_to="none",
        dataloader_num_workers=4,
        group_by_length=True,
    )

    trainer = Trainer(
        model=model_lora,
        args=training_args,
        train_dataset=tokenized_dataset,
        data_collator=DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)
    )

    t0 = time.time()
    trainer.train()
    print(f"Обучение завершено за {(time.time()-t0)/60:.1f} мин")

    trainer.save_model(output_dir)

    # Возвращаем базовую модель без LoRA весов (важно для следующего эксперимента)
    model_lora = model_lora.unload()
    torch.cuda.empty_cache()

    return output_dir

In [ ]:
import glob
import re
from peft import PeftModel

def evaluate_checkpoints_on_val(output_dir, val_df, feature_names, target_name,
                                 prompt_config, tokenizer, base_model, device,
                                 eval_missing_rate=0.0):

    checkpoints = sorted(
        glob.glob(f"{output_dir}/checkpoint-*"),
        key=lambda x: int(re.search(r'checkpoint-(\d+)', x).group(1))
    )
    print(f"Найдено {len(checkpoints)} checkpoints в {output_dir}")

    best_auc, best_checkpoint = 0.0, None
    for cp in checkpoints:
        model_eval = PeftModel.from_pretrained(base_model, cp).eval()

        y_true, y_prob = [], []
        for _, row in tqdm(val_df.iterrows(), total=len(val_df),
                           desc=f"Val eval {cp.split('/')[-1]}"):
            prompt = create_prompt_missing(
                row, feature_names, target_name, prompt_config, tokenizer,
                missing_rate=eval_missing_rate)
            _, prob = predict_single_with_prob(prompt, prompt_config, model_eval, tokenizer, device)
            y_true.append(row[target_name])
            y_prob.append(prob)

        y_pred = [1 if p > 0.5 else 0 for p in y_prob]
        auc = roc_auc_score(y_true, y_prob)
        print(f"  {cp.split('/')[-1]}  ROC-AUC={auc:.4f}")

        if auc > best_auc:
            best_auc = auc
            best_checkpoint = cp

        del model_eval
        torch.cuda.empty_cache()

    print(f"Лучший checkpoint: {best_checkpoint}  (ROC-AUC={best_auc:.4f})")
    return best_checkpoint

In [ ]:
def evaluate_on_test(best_checkpoint, test_df, feature_names, target_name,
                     prompt_config, tokenizer, base_model, device,
                     eval_missing_rate=0.0):
    model_finetuned = PeftModel.from_pretrained(base_model, best_checkpoint).eval()

    y_true, y_pred, y_prob = [], [], []
    t0 = time.time()

    for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc="Test eval"):
        prompt = create_prompt_missing(
            row, feature_names, target_name, prompt_config, tokenizer,
            missing_rate=eval_missing_rate)
        response, prob = predict_single_with_prob(
            prompt, prompt_config, model_finetuned, tokenizer, device)
        y_true.append(row[target_name])
        y_pred.append(parse_prediction(response, prompt_config))
        y_prob.append(prob)

    elapsed = time.time() - t0
    metrics = compute_metrics(np.array(y_true), np.array(y_pred), np.array(y_prob))
    boot = bootstrap_metrics(np.array(y_true), np.array(y_pred), np.array(y_prob), n_iter=1000)

    print("Метрики:")
    for k, v in metrics.items():
        print(f"  {k}: {v:.4f}")
    print("Bootstrap (mean±std):")
    for k, v in boot.items():
        print(f"  {k}: {v}")

    del model_finetuned
    torch.cuda.empty_cache()

    return {
        "metrics": metrics,
        "bootstrap": boot,
        "time_total": elapsed,
        "time_per_sample": elapsed / len(y_true),
        "best_checkpoint": best_checkpoint,
        "eval_missing_rate": eval_missing_rate,
    }

In [ ]:
all_results = {}

start_time = time.time()

for missing_rate in MISSING_RATES:
    tag = f"missing_{int(missing_rate * 100):03d}pct"
    print(f"Эксперимент: missing_rate = {missing_rate:.0%}")

    # 7.1 Fine-tuning
    output_dir = finetune_model(
        missing_rate=missing_rate,
        train_df_balanced=train_df_balanced,
        feature_names=feature_names,
        target_name=target_name,
        prompt_config=prompt_config,
        tokenizer=tokenizer,
        base_model=base_model,
        device=device,
        num_epochs=20,
        batch_size=16,
    )

    # 7.2 Выбор лучшего checkpoint по val
    best_cp = evaluate_checkpoints_on_val(
        output_dir=output_dir,
        val_df=val_df,
        feature_names=feature_names,
        target_name=target_name,
        prompt_config=prompt_config,
        tokenizer=tokenizer,
        base_model=base_model,
        device=device,
        eval_missing_rate=missing_rate,
    )

    # 7.3 Оценка на test
    result = evaluate_on_test(
        best_checkpoint=best_cp,
        test_df=test_df,
        feature_names=feature_names,
        target_name=target_name,
        prompt_config=prompt_config,
        tokenizer=tokenizer,
        base_model=base_model,
        device=device,
        eval_missing_rate=missing_rate,
    )

    all_results[tag] = result

print(f"Эксперимент завершен за {(time.time()-start_time)/60:.1f} мин")

Эксперимент: missing_rate = 0%
  Fine-tuning: missing_rate = 0%
  Output dir : /content/drive/MyDrive/finetuned_qwen_missing_Credit_G_missing000


Dataset (missing=0%):   0%|          | 0/840 [00:00<?, ?it/s]

Map:   0%|          | 0/840 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:285: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


trainable params: 33,030,144 || all params: 4,055,498,240 || trainable%: 0.8145


...

Val eval checkpoint-540:   0%|          | 0/200 [00:00<?, ?it/s]

  checkpoint-540  ROC-AUC=0.4942
Лучший checkpoint: /content/drive/MyDrive/finetuned_qwen_missing_Credit_G_missing090/checkpoint-270  (ROC-AUC=0.6246)


/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:285: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


Test eval:   0%|          | 0/200 [00:00<?, ?it/s]

Метрики:
  ROC-AUC: 0.5092
  F1: 0.2013
  Accuracy: 0.3650
  Precision: 0.8421
  Recall: 0.1143
Bootstrap (mean±std):
  ROC-AUC: 0.5093±0.0405
  F1: 0.2023±0.0399
  Accuracy: 0.3666±0.0340
  Precision: 0.8432±0.0823
  Recall: 0.1155±0.0254
Эксперимент завершен за 109.6 мин


In [ ]:
all_results

{'missing_000pct': {'metrics': {'ROC-AUC': np.float64(0.7323809523809524),
   'F1': 0.7154471544715447,
   'Accuracy': 0.65,
   'Precision': 0.8301886792452831,
   'Recall': 0.6285714285714286},
  'bootstrap': {'ROC-AUC': '0.7320±0.0380',
   'F1': '0.7138±0.0315',
   'Accuracy': '0.6496±0.0325',
   'Precision': '0.8298±0.0365',
   'Recall': '0.6276±0.0399'},
  'time_total': 45.399985551834106,
  'time_per_sample': 0.22699992775917052,
  'best_checkpoint': '/content/drive/MyDrive/finetuned_qwen_missing_Credit_G_missing000/checkpoint-270',
  'eval_missing_rate': 0.0},
 'missing_020pct': {'metrics': {'ROC-AUC': np.float64(0.6995238095238095),
   'F1': 0.7107438016528925,
   'Accuracy': 0.65,
   'Precision': 0.8431372549019608,
   'Recall': 0.6142857142857143},
  'bootstrap': {'ROC-AUC': '0.6987±0.0395',
   'F1': '0.7085±0.0337',
   'Accuracy': '0.6489±0.0346',
   'Precision': '0.8420±0.0368',
   'Recall': '0.6128±0.0418'},
  'time_total': 45.273927450180054,
  'time_per_sample': 0.2263696

In [ ]:
print("\nBootstrap (mean±std):")
for tag, res in all_results.items():
    rate = tag.replace("missing_", "").replace("pct", "%").lstrip("0") or "0%"
    print(f"\n  missing={rate}")
    for k, v in res["bootstrap"].items():
        print(f"    {k}: {v}")


Bootstrap (mean±std):

  missing=%
    ROC-AUC: 0.7320±0.0380
    F1: 0.7138±0.0315
    Accuracy: 0.6496±0.0325
    Precision: 0.8298±0.0365
    Recall: 0.6276±0.0399

  missing=20%
    ROC-AUC: 0.6987±0.0395
    F1: 0.7085±0.0337
    Accuracy: 0.6489±0.0346
    Precision: 0.8420±0.0368
    Recall: 0.6128±0.0418

  missing=50%
    ROC-AUC: 0.6045±0.0410
    F1: 0.6853±0.0323
    Accuracy: 0.5893±0.0343
    Precision: 0.7366±0.0396
    Recall: 0.6422±0.0403

  missing=90%
    ROC-AUC: 0.5093±0.0405
    F1: 0.2023±0.0399
    Accuracy: 0.3666±0.0340
    Precision: 0.8432±0.0823
    Recall: 0.1155±0.0254


ROC-AUC (0.73): Умеренный уровень производительности. Задача классификации кредитных рисков традиционно является сложной для текстовых моделей. При 20% пропусков показатель снижается до 0.70, а при критических 90% падает до 0.51, что фактически соответствует уровню случайного угадывания.

Precision (0.83): Очень высокий показатель точности. В отличие от предыдущих задач, здесь модель проявляет «осторожность»: она редко помечает клиента как надежного, если нет достаточных оснований. Примечательно, что даже при 90% пропусков точность удерживается на уровне 0.84, что говорит о жесткой фильтрации рисков моделью.

F1-score (0.71): Стабильный баланс между точностью и полнотой на уровнях пропусков до 50%. Это указывает на то, что модель успешно извлекает ключевые финансовые предикторы даже из неполных анкет. Однако при 90% пропусков происходит обвал до 0.20, так как модель теряет способность идентифицировать большинство положительных кейсов.

Recall (0.63): Относительно низкая полнота. Модель пропускает значительную часть потенциально «хороших» заемщиков, отдавая приоритет минимизации рисков (высокий Precision). При экстремальных пропусках (90%) полнота падает до критических 0.12, что делает использование модели в таком режиме нецелесообразным.

Время эксперимента: ~ 1 ч 40 мин.

Использовано оперативная памяти графического процесса: 50/80GB.

Графический процессор A100 80GB.